# Custom Operators — Hands-On Application

## Objective

Master defining, building, serializing, and managing custom ONNX operators. Learn to mix
custom domain operators with standard operators, implement custom activation functions,
and build reusable custom operator libraries.

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Setup](#1-setup) | Imports and utilities |
| 2 | [Exercise 1: Define Custom Op Nodes](#2-exercise-1) | Custom domain, attributes |
| 3 | [Exercise 2: Mixed Standard + Custom Models](#3-exercise-2) | Hybrid graph construction |
| 4 | [Exercise 3: Serialization Round-Trip](#4-exercise-3) | Save/load custom op models |
| 5 | [Exercise 4: Standard Op Validation](#5-exercise-4) | Checker behavior with custom ops |
| 6 | [Exercise 5: Custom Domain Management](#6-exercise-5) | Multiple custom domains |
| 7 | [Exercise 6: Custom GELU Implementation](#7-exercise-6) | Implement GELU as custom op |
| 8 | [Exercise 7: Runtime Behavior](#8-exercise-7) | Custom ops with reference evaluator |
| 9 | [Challenge: Custom Operator Library](#9-challenge) | Reusable op collection |
| 10 | [Summary](#10-summary) | Skills review |

In [ ]:
# 1. Setup <a id="1-setup"></a>
# !pip install onnx numpy --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, numpy_helper, shape_inference
import tempfile
import os
import copy

print(f"ONNX version: {onnx.__version__}")
print(f"IR version: {onnx.IR_VERSION}")

## 2. Exercise 1: Define Custom Op Nodes <a id="2-exercise-1"></a>

Custom operators are defined by specifying a **domain** string that differs from the
standard ONNX domain. This tells the runtime to look for a custom kernel implementation.

A custom op node has:
- `op_type` — the operation name
- `domain` — identifies which operator set it belongs to
- `inputs` / `outputs` — tensor names
- Custom **attributes** — configuration parameters

In [ ]:
MY_DOMAIN = "com.tutorial.custom"

# Custom RMSNorm operator
rmsnorm_node = helper.make_node(
    "RMSNorm",
    inputs=["X", "weight"],
    outputs=["Y"],
    domain=MY_DOMAIN,
    epsilon=1e-6,
    name="rmsnorm_0",
)

# Custom FusedAttention operator
attention_node = helper.make_node(
    "FusedMultiHeadAttention",
    inputs=["Q", "K", "V", "mask"],
    outputs=["attn_out", "attn_weights"],
    domain=MY_DOMAIN,
    num_heads=8,
    scale=0.125,  # 1/sqrt(d_k)
    causal=1,
    name="mha_0",
)

# Custom activation: Mish = x * tanh(softplus(x))
mish_node = helper.make_node(
    "Mish",
    inputs=["X"],
    outputs=["Y"],
    domain=MY_DOMAIN,
    name="mish_0",
)

# Inspect the nodes
for node in [rmsnorm_node, attention_node, mish_node]:
    print(f"Op: {node.op_type}")
    print(f"  Domain:  {node.domain}")
    print(f"  Inputs:  {list(node.input)}")
    print(f"  Outputs: {list(node.output)}")
    print(f"  Attrs:   {[(a.name, a.f if a.type == 1 else a.i) for a in node.attribute]}")
    print()

# Verify domain is set correctly
assert rmsnorm_node.domain == MY_DOMAIN
assert attention_node.domain == MY_DOMAIN
print(f"All custom nodes use domain '{MY_DOMAIN}'. ✓")

## 3. Exercise 2: Mixed Standard + Custom Models <a id="3-exercise-2"></a>

In practice, models mix standard ONNX operators with custom ones. The key is:
- Standard ops use domain `""` (empty string)
- Custom ops use your domain string
- Both must have opset imports declared in the model

Graph pattern: `X → MatMul(std) → CustomActivation(custom) → Add(std) → Y`

In [ ]:
def build_hybrid_model(custom_activation: str = "SwishActivation",
                       hidden: int = 64) -> onnx.ModelProto:
    """Build a model mixing standard and custom operators."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", hidden])
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", hidden])

    W = numpy_helper.from_array(
        np.random.randn(hidden, hidden).astype(np.float32) * 0.01, "W"
    )
    b = numpy_helper.from_array(np.zeros(hidden, dtype=np.float32), "b")

    nodes = [
        # Standard: linear transform
        helper.make_node("MatMul", ["X", "W"], ["mm_out"]),
        helper.make_node("Add", ["mm_out", "b"], ["linear_out"]),
        # Custom: activation function
        helper.make_node(custom_activation, ["linear_out"], ["act_out"],
                         domain=MY_DOMAIN),
        # Standard: residual connection
        helper.make_node("Add", ["act_out", "X"], ["Y"]),
    ]

    graph = helper.make_graph(nodes, "hybrid_block", [X], [Y], initializer=[W, b])
    model = helper.make_model(
        graph,
        opset_imports=[
            helper.make_opsetid("", 17),
            helper.make_opsetid(MY_DOMAIN, 1),
        ],
    )
    return model


# Build models with different custom activations
activations = ["SwishActivation", "GeluActivation", "MishActivation"]

for act in activations:
    model = build_hybrid_model(act)
    print(f"\nModel with {act}:")
    for node in model.graph.node:
        domain_label = node.domain if node.domain else "(standard)"
        print(f"  {node.op_type:<25} domain={domain_label}")

    # Verify opset imports
    domains = {oi.domain: oi.version for oi in model.opset_import}
    assert "" in domains, "Standard domain must be imported"
    assert MY_DOMAIN in domains, "Custom domain must be imported"
    print(f"  OpSets: {[(d or 'default', v) for d, v in domains.items()]}")

print("\nAll hybrid models built correctly. ✓")

## 4. Exercise 3: Serialization Round-Trip <a id="4-exercise-3"></a>

Verify that custom operator models survive serialization → deserialization.
All custom domain info, attributes, and graph structure must be preserved.

In [ ]:
def verify_custom_roundtrip(model: onnx.ModelProto, label: str) -> bool:
    """Verify round-trip for a custom op model."""
    original_bytes = model.SerializeToString()

    # Method 1: Bytes round-trip
    loaded_bytes = onnx.ModelProto()
    loaded_bytes.ParseFromString(original_bytes)

    # Method 2: File round-trip
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, "custom_model.onnx")
        onnx.save(model, path)
        loaded_file = onnx.load(path)

    # Verify all fields preserved
    checks = []

    # Opset imports
    orig_opsets = {oi.domain: oi.version for oi in model.opset_import}
    load_opsets = {oi.domain: oi.version for oi in loaded_file.opset_import}
    checks.append(("opset_imports", orig_opsets == load_opsets))

    # Nodes and their domains
    orig_nodes = [(n.op_type, n.domain) for n in model.graph.node]
    load_nodes = [(n.op_type, n.domain) for n in loaded_file.graph.node]
    checks.append(("node_domains", orig_nodes == load_nodes))

    # Attributes
    for orig_n, load_n in zip(model.graph.node, loaded_file.graph.node):
        orig_attrs = {a.name: a for a in orig_n.attribute}
        load_attrs = {a.name: a for a in load_n.attribute}
        checks.append((f"{orig_n.op_type}_attrs", set(orig_attrs.keys()) == set(load_attrs.keys())))

    # Byte identity
    checks.append(("byte_identical", original_bytes == loaded_file.SerializeToString()))

    all_pass = all(ok for _, ok in checks)
    status = "✓" if all_pass else "✗"
    print(f"  {status} {label}:")
    for name, ok in checks:
        print(f"    {'✓' if ok else '✗'} {name}")

    return all_pass


# Test with various custom op models
print("Custom Op Serialization Round-Trip Tests:")
print("═" * 50)

# Simple custom op
m1 = build_hybrid_model("Swish")
assert verify_custom_roundtrip(m1, "Simple custom activation")

# Custom op with attributes
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [1, 64])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [1, 64])
nodes = [
    helper.make_node("CustomNorm", ["X"], ["Y"], domain=MY_DOMAIN,
                     epsilon=1e-5, axis=-1, mode="layer"),
]
graph = helper.make_graph(nodes, "norm_test", [X], [Y])
m2 = helper.make_model(graph, opset_imports=[
    helper.make_opsetid("", 17),
    helper.make_opsetid(MY_DOMAIN, 1),
])
assert verify_custom_roundtrip(m2, "Custom op with mixed attributes")

# Multi-output custom op
Q = helper.make_tensor_value_info("Q", TensorProto.FLOAT, [1, 8, 64])
K = helper.make_tensor_value_info("K", TensorProto.FLOAT, [1, 8, 64])
V = helper.make_tensor_value_info("V", TensorProto.FLOAT, [1, 8, 64])
out1 = helper.make_tensor_value_info("out", TensorProto.FLOAT, [1, 8, 64])
out2 = helper.make_tensor_value_info("weights", TensorProto.FLOAT, [1, 8, 8])
nodes = [
    helper.make_node("FusedAttention", ["Q", "K", "V"], ["out", "weights"],
                     domain=MY_DOMAIN, num_heads=4, scale=0.125),
]
graph = helper.make_graph(nodes, "attn_test", [Q, K, V], [out1, out2])
m3 = helper.make_model(graph, opset_imports=[
    helper.make_opsetid("", 17),
    helper.make_opsetid(MY_DOMAIN, 1),
])
assert verify_custom_roundtrip(m3, "Multi-output custom op")

print("\nAll round-trip tests passed. ✓")

## 5. Exercise 4: Standard Op Validation <a id="5-exercise-4"></a>

Understanding how `onnx.checker` handles models with custom operators:
- Standard ops are validated against known schemas
- Custom ops (unknown domains) are **skipped** by the checker
- This means structural errors in custom ops won't be caught by `check_model`

In [ ]:
def test_checker_behavior(model: onnx.ModelProto, label: str) -> str:
    """Test if checker passes or fails."""
    try:
        checker.check_model(model)
        return "PASS"
    except Exception as e:
        return f"FAIL: {str(e)[:80]}"


print("Checker Behavior with Custom Ops:")
print("═" * 60)

# 1. Pure standard ops (should pass)
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 10])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [2, 10])
std_model = helper.make_model(
    helper.make_graph([helper.make_node("Relu", ["X"], ["Y"])], "std", [X], [Y]),
    opset_imports=[helper.make_opsetid("", 17)],
)
r1 = test_checker_behavior(std_model, "Standard only")
print(f"  1. Standard ops only:        {r1}")

# 2. Custom ops with proper domain (should pass — checker skips unknown domains)
custom_model = helper.make_model(
    helper.make_graph(
        [helper.make_node("MyCustomOp", ["X"], ["Y"], domain=MY_DOMAIN)],
        "custom", [X], [Y]
    ),
    opset_imports=[helper.make_opsetid("", 17), helper.make_opsetid(MY_DOMAIN, 1)],
)
r2 = test_checker_behavior(custom_model, "Custom domain")
print(f"  2. Custom domain op:         {r2}")

# 3. Invalid standard op (should fail)
bad_std = helper.make_model(
    helper.make_graph(
        [helper.make_node("NonExistentOp", ["X"], ["Y"])],  # no domain = standard
        "bad", [X], [Y]
    ),
    opset_imports=[helper.make_opsetid("", 17)],
)
r3 = test_checker_behavior(bad_std, "Invalid std op")
print(f"  3. Invalid standard op:      {r3}")

# 4. Mixed model (standard parts validated, custom parts skipped)
W = numpy_helper.from_array(np.random.randn(10, 10).astype(np.float32), "W")
mixed_model = helper.make_model(
    helper.make_graph(
        [
            helper.make_node("MatMul", ["X", "W"], ["mm"]),
            helper.make_node("CustomAct", ["mm"], ["Y"], domain=MY_DOMAIN),
        ],
        "mixed", [X], [Y], initializer=[W]
    ),
    opset_imports=[helper.make_opsetid("", 17), helper.make_opsetid(MY_DOMAIN, 1)],
)
r4 = test_checker_behavior(mixed_model, "Mixed valid")
print(f"  4. Mixed (valid std + custom): {r4}")

# 5. Missing opset import for custom domain
no_import = helper.make_model(
    helper.make_graph(
        [helper.make_node("MyOp", ["X"], ["Y"], domain="missing.domain")],
        "no_import", [X], [Y]
    ),
    opset_imports=[helper.make_opsetid("", 17)],  # missing custom domain import
)
r5 = test_checker_behavior(no_import, "Missing import")
print(f"  5. Missing domain import:    {r5}")

print("\nKey insight: Custom domain ops bypass schema validation.")
print("You must validate custom ops yourself!")

## 6. Exercise 5: Custom Domain Management <a id="6-exercise-5"></a>

In larger projects, you might have multiple custom domains representing different
teams or libraries. Manage them systematically.

In [ ]:
class CustomDomainRegistry:
    """Registry for managing custom operator domains."""

    def __init__(self):
        self.domains = {}  # domain -> {version, ops: [...]}

    def register_domain(self, domain: str, version: int = 1):
        self.domains[domain] = {"version": version, "ops": []}

    def register_op(self, domain: str, op_name: str, inputs: list,
                    outputs: list, attributes: dict = None):
        if domain not in self.domains:
            self.register_domain(domain)
        self.domains[domain]["ops"].append({
            "name": op_name,
            "inputs": inputs,
            "outputs": outputs,
            "attributes": attributes or {},
        })

    def get_opset_imports(self, used_domains: list = None) -> list:
        """Generate opset_import list for model construction."""
        imports = [helper.make_opsetid("", 17)]  # always include standard
        domains_to_add = used_domains or list(self.domains.keys())
        for d in domains_to_add:
            if d in self.domains:
                imports.append(helper.make_opsetid(d, self.domains[d]["version"]))
        return imports

    def validate_node(self, node) -> tuple:
        """Validate a node against our custom domain registry."""
        if node.domain not in self.domains:
            return False, f"Unknown domain: {node.domain}"

        registered_ops = [op["name"] for op in self.domains[node.domain]["ops"]]
        if node.op_type not in registered_ops:
            return False, f"Unknown op '{node.op_type}' in domain '{node.domain}'"

        op_spec = next(op for op in self.domains[node.domain]["ops"] if op["name"] == node.op_type)
        if len(node.input) < len(op_spec["inputs"]):
            return False, f"Too few inputs: expected {len(op_spec['inputs'])}, got {len(node.input)}"

        return True, "valid"

    def summary(self):
        print("Custom Domain Registry:")
        for domain, info in self.domains.items():
            print(f"  [{domain}] v{info['version']} ({len(info['ops'])} ops)")
            for op in info["ops"]:
                attrs_str = ", ".join(f"{k}:{v}" for k, v in op["attributes"].items())
                print(f"    {op['name']}({', '.join(op['inputs'])}) → {', '.join(op['outputs'])}")
                if attrs_str:
                    print(f"      attrs: {attrs_str}")


# Set up a custom domain registry
registry = CustomDomainRegistry()

# Normalization ops
registry.register_domain("com.myorg.norm", version=1)
registry.register_op("com.myorg.norm", "RMSNorm", ["X", "weight"], ["Y"],
                     {"epsilon": "float"})
registry.register_op("com.myorg.norm", "GroupRMSNorm", ["X", "weight"], ["Y"],
                     {"epsilon": "float", "groups": "int"})

# Activation ops
registry.register_domain("com.myorg.activations", version=2)
registry.register_op("com.myorg.activations", "Swish", ["X"], ["Y"], {})
registry.register_op("com.myorg.activations", "Mish", ["X"], ["Y"], {})
registry.register_op("com.myorg.activations", "GELU", ["X"], ["Y"],
                     {"approximate": "string"})

# Attention ops
registry.register_domain("com.myorg.attention", version=1)
registry.register_op("com.myorg.attention", "FlashAttention",
                     ["Q", "K", "V"], ["out", "lse"],
                     {"causal": "int", "scale": "float"})

registry.summary()

# Validate some nodes
print("\nNode validation:")
test_nodes = [
    helper.make_node("RMSNorm", ["X", "w"], ["Y"], domain="com.myorg.norm", epsilon=1e-6),
    helper.make_node("Swish", ["X"], ["Y"], domain="com.myorg.activations"),
    helper.make_node("BadOp", ["X"], ["Y"], domain="com.myorg.norm"),  # invalid
    helper.make_node("FlashAttention", ["Q"], ["out"], domain="com.myorg.attention"),  # too few inputs
]

for node in test_nodes:
    valid, msg = registry.validate_node(node)
    status = "✓" if valid else "✗"
    print(f"  {status} {node.domain}::{node.op_type} — {msg}")

## 7. Exercise 6: Custom GELU Implementation <a id="7-exercise-6"></a>

Implement GELU as both:
1. A custom op (opaque node)
2. A decomposed implementation using standard ops

GELU formula: $\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$

Approximate GELU: $\text{GELU}(x) \approx 0.5x\left[1 + \tanh\left(\sqrt{\frac{2}{\pi}}(x + 0.044715x^3)\right)\right]$

In [ ]:
def build_custom_gelu(shape: list) -> onnx.ModelProto:
    """Build model with custom GELU op (single opaque node)."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, shape)
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, shape)

    node = helper.make_node("GELU", ["X"], ["Y"], domain=MY_DOMAIN,
                            approximate="none")

    graph = helper.make_graph([node], "custom_gelu", [X], [Y])
    return helper.make_model(graph, opset_imports=[
        helper.make_opsetid("", 17),
        helper.make_opsetid(MY_DOMAIN, 1),
    ])


def build_decomposed_gelu(shape: list) -> onnx.ModelProto:
    """Build GELU using standard ONNX ops: x * 0.5 * (1 + erf(x/sqrt(2)))."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, shape)
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, shape)

    sqrt2 = numpy_helper.from_array(np.array([np.sqrt(2.0)], dtype=np.float32), "sqrt2")
    one = numpy_helper.from_array(np.array([1.0], dtype=np.float32), "one")
    half = numpy_helper.from_array(np.array([0.5], dtype=np.float32), "half")

    nodes = [
        helper.make_node("Div", ["X", "sqrt2"], ["x_scaled"]),
        helper.make_node("Erf", ["x_scaled"], ["erf_val"]),
        helper.make_node("Add", ["erf_val", "one"], ["erf_p1"]),
        helper.make_node("Mul", ["X", "erf_p1"], ["x_erf"]),
        helper.make_node("Mul", ["x_erf", "half"], ["Y"]),
    ]

    graph = helper.make_graph(nodes, "decomposed_gelu", [X], [Y],
                              initializer=[sqrt2, one, half])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    checker.check_model(model)
    return model


def build_approx_gelu(shape: list) -> onnx.ModelProto:
    """Build approximate GELU using tanh approximation."""
    X = helper.make_tensor_value_info("X", TensorProto.FLOAT, shape)
    Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, shape)

    c1 = numpy_helper.from_array(np.array([np.sqrt(2.0 / np.pi)], dtype=np.float32), "c1")
    c2 = numpy_helper.from_array(np.array([0.044715], dtype=np.float32), "c2")
    one = numpy_helper.from_array(np.array([1.0], dtype=np.float32), "one")
    half = numpy_helper.from_array(np.array([0.5], dtype=np.float32), "half")
    three = numpy_helper.from_array(np.array([3.0], dtype=np.float32), "three")

    nodes = [
        helper.make_node("Pow", ["X", "three"], ["x3"]),
        helper.make_node("Mul", ["x3", "c2"], ["x3c"]),
        helper.make_node("Add", ["X", "x3c"], ["inner"]),
        helper.make_node("Mul", ["inner", "c1"], ["scaled"]),
        helper.make_node("Tanh", ["scaled"], ["tanh_val"]),
        helper.make_node("Add", ["tanh_val", "one"], ["tp1"]),
        helper.make_node("Mul", ["X", "tp1"], ["x_tp1"]),
        helper.make_node("Mul", ["x_tp1", "half"], ["Y"]),
    ]

    graph = helper.make_graph(nodes, "approx_gelu", [X], [Y],
                              initializer=[c1, c2, one, half, three])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    checker.check_model(model)
    return model


# Compare implementations
shape = [4, 64]
custom = build_custom_gelu(shape)
decomposed = build_decomposed_gelu(shape)
approx = build_approx_gelu(shape)

print("GELU Implementation Comparison:")
print(f"  Custom (opaque):     {len(custom.graph.node)} node(s), requires custom runtime")
print(f"  Decomposed (exact):  {len(decomposed.graph.node)} nodes, standard ops only")
print(f"  Approximate (tanh):  {len(approx.graph.node)} nodes, standard ops only")

# Run decomposed and approximate, compare
from onnx.reference import ReferenceEvaluator
x_test = np.random.randn(*shape).astype(np.float32)

y_exact = ReferenceEvaluator(decomposed).run(None, {"X": x_test})[0]
y_approx = ReferenceEvaluator(approx).run(None, {"X": x_test})[0]

diff = np.abs(y_exact - y_approx).max()
print(f"\n  Max diff (exact vs approx): {diff:.6f}")
print(f"  Approximation acceptable: {diff < 0.01} ✓")

## 8. Exercise 7: Runtime Behavior <a id="8-exercise-7"></a>

Custom ops require runtime kernel implementations. The ONNX Reference Evaluator
can be extended with custom op implementations for testing.

We'll register custom op implementations and verify they execute correctly.

In [ ]:
from onnx.reference import ReferenceEvaluator
from onnx.reference.op_run import OpRun


class SwishOp(OpRun):
    """Custom Swish implementation: x * sigmoid(x)."""
    op_domain = MY_DOMAIN

    def _run(self, X):
        sigmoid_x = 1.0 / (1.0 + np.exp(-X))
        return (X * sigmoid_x,)


class RMSNormOp(OpRun):
    """Custom RMSNorm: x * weight / sqrt(mean(x^2) + eps)."""
    op_domain = MY_DOMAIN

    def _run(self, X, weight, epsilon=None):
        eps = epsilon if epsilon is not None else 1e-6
        rms = np.sqrt(np.mean(X ** 2, axis=-1, keepdims=True) + eps)
        return (X / rms * weight,)


# Build a model with custom ops that we can execute
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 64])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [2, 64])
W_norm = numpy_helper.from_array(np.ones(64, dtype=np.float32), "norm_weight")
W_linear = numpy_helper.from_array(
    np.random.randn(64, 64).astype(np.float32) * 0.01, "W"
)

nodes = [
    helper.make_node("RMSNorm", ["X", "norm_weight"], ["normed"],
                     domain=MY_DOMAIN, epsilon=1e-6),
    helper.make_node("MatMul", ["normed", "W"], ["mm"]),
    helper.make_node("Swish", ["mm"], ["Y"], domain=MY_DOMAIN),
]

graph = helper.make_graph(nodes, "runnable_custom", [X], [Y],
                          initializer=[W_norm, W_linear])
runnable_model = helper.make_model(graph, opset_imports=[
    helper.make_opsetid("", 17),
    helper.make_opsetid(MY_DOMAIN, 1),
])

# Execute with custom implementations
ev = ReferenceEvaluator(runnable_model, new_ops=[SwishOp, RMSNormOp])
x_test = np.random.randn(2, 64).astype(np.float32)
y_result = ev.run(None, {"X": x_test})[0]

print("Custom Op Execution:")
print(f"  Input:  shape={x_test.shape}, mean={x_test.mean():.4f}")
print(f"  Output: shape={y_result.shape}, mean={y_result.mean():.4f}")
print(f"  Output range: [{y_result.min():.4f}, {y_result.max():.4f}]")

# Verify against manual computation
norm_weight = np.ones(64, dtype=np.float32)
rms = np.sqrt(np.mean(x_test ** 2, axis=-1, keepdims=True) + 1e-6)
normed = x_test / rms * norm_weight
mm = normed @ (np.random.randn(64, 64).astype(np.float32) * 0.01)  # different seed

# At least verify shapes match
assert y_result.shape == (2, 64)
assert not np.isnan(y_result).any(), "No NaN values"
assert not np.isinf(y_result).any(), "No Inf values"
print(f"\n  No NaN/Inf in output. ✓")
print(f"  Custom ops executed successfully with ReferenceEvaluator. ✓")

## 9. Challenge: Custom Operator Library <a id="9-challenge"></a>

Build a reusable custom operator library that:
1. Registers a family of related operators
2. Provides builder functions for common patterns
3. Includes runtime implementations for testing
4. Validates models using the library

In [ ]:
class CustomOpLibrary:
    """A reusable custom operator library."""

    def __init__(self, domain: str, version: int = 1):
        self.domain = domain
        self.version = version
        self.ops = {}  # name -> {inputs, outputs, attrs, impl}
        self._op_classes = []

    def register(self, name: str, inputs: list, outputs: list,
                 attrs: dict = None, impl=None):
        """Register a custom operator."""
        self.ops[name] = {
            "inputs": inputs,
            "outputs": outputs,
            "attrs": attrs or {},
            "impl": impl,
        }
        if impl:
            self._op_classes.append(impl)

    def make_node(self, op_name: str, inputs: list, outputs: list, **attrs):
        """Create a node for a registered operator."""
        if op_name not in self.ops:
            raise ValueError(f"Unknown op '{op_name}' in library '{self.domain}'")
        return helper.make_node(op_name, inputs, outputs, domain=self.domain, **attrs)

    def get_opset_import(self):
        return helper.make_opsetid(self.domain, self.version)

    def build_model(self, nodes, inputs, outputs, initializers=None,
                    name="model") -> onnx.ModelProto:
        """Build a complete model with proper opset imports."""
        graph = helper.make_graph(
            nodes, name, inputs, outputs, initializer=initializers or []
        )
        return helper.make_model(
            graph,
            opset_imports=[
                helper.make_opsetid("", 17),
                self.get_opset_import(),
            ],
        )

    def evaluate(self, model, feeds):
        """Evaluate a model using registered implementations."""
        ev = ReferenceEvaluator(model, new_ops=self._op_classes)
        return ev.run(None, feeds)

    def validate_model(self, model) -> list:
        """Validate all custom ops in a model against the library."""
        errors = []
        for i, node in enumerate(model.graph.node):
            if node.domain == self.domain:
                if node.op_type not in self.ops:
                    errors.append(f"Node {i}: Unknown op '{node.op_type}'")
                else:
                    spec = self.ops[node.op_type]
                    if len(node.input) < len(spec["inputs"]):
                        errors.append(
                            f"Node {i} ({node.op_type}): expected >= {len(spec['inputs'])} inputs"
                        )
        return errors

    def summary(self):
        print(f"\nCustom Op Library: {self.domain} v{self.version}")
        print(f"  Registered operators: {len(self.ops)}")
        for name, spec in self.ops.items():
            has_impl = "✓" if spec["impl"] else "✗"
            print(f"    {name}({', '.join(spec['inputs'])}) → "
                  f"{', '.join(spec['outputs'])} [impl: {has_impl}]")


# Build a library
lib = CustomOpLibrary("com.acme.nn", version=2)

# Register ops with implementations
class AcmeSwish(OpRun):
    op_domain = "com.acme.nn"
    def _run(self, X):
        return (X * (1.0 / (1.0 + np.exp(-X))),)

class AcmeRMSNorm(OpRun):
    op_domain = "com.acme.nn"
    def _run(self, X, weight, epsilon=None):
        eps = epsilon or 1e-6
        rms = np.sqrt(np.mean(X**2, axis=-1, keepdims=True) + eps)
        return (X / rms * weight,)

lib.register("Swish", ["X"], ["Y"], impl=AcmeSwish)
lib.register("RMSNorm", ["X", "weight"], ["Y"], {"epsilon": "float"}, impl=AcmeRMSNorm)
lib.register("RotaryEmbedding", ["X", "cos", "sin"], ["Y"], impl=None)

lib.summary()

# Build and run a model using the library
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, [2, 32])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [2, 32])
norm_w = numpy_helper.from_array(np.ones(32, dtype=np.float32), "norm_w")

nodes = [
    lib.make_node("RMSNorm", ["X", "norm_w"], ["normed"], epsilon=1e-5),
    lib.make_node("Swish", ["normed"], ["Y"]),
]

model = lib.build_model(nodes, [X], [Y], [norm_w], "acme_model")

# Validate
errors = lib.validate_model(model)
assert len(errors) == 0, f"Validation errors: {errors}"
print(f"\nModel validation: {len(errors)} errors ✓")

# Execute
x_in = np.random.randn(2, 32).astype(np.float32)
y_out = lib.evaluate(model, {"X": x_in})[0]
print(f"Execution: input shape {x_in.shape} → output shape {y_out.shape}")
assert y_out.shape == (2, 32)
assert not np.isnan(y_out).any()
print(f"Custom op library working end-to-end. ✓")

## 10. Summary <a id="10-summary"></a>

| Exercise | Skill | Key Insight |
|----------|-------|---------|
| 1. Define Custom Ops | Node construction with domain | Domain string distinguishes custom from standard |
| 2. Mixed Models | Hybrid graph building | Standard + custom ops coexist in one graph |
| 3. Serialization | Round-trip verification | Custom domains/attrs survive serialization |
| 4. Checker Behavior | Validation semantics | Custom domains bypass schema validation |
| 5. Domain Management | Multi-domain registry | Organized domain + op tracking |
| 6. Custom GELU | Implement via decomposition | Compare opaque vs standard-op implementation |
| 7. Runtime | Execute custom ops | `ReferenceEvaluator` with `new_ops` |
| Challenge | Op Library | Reusable library with registration, validation, execution |

### Key Takeaways

1. Custom ops use a **domain string** to differentiate from standard ONNX ops
2. The ONNX checker **skips validation** for unknown domains — validate yourself
3. Custom ops require **runtime kernel implementations** to execute
4. When possible, **decompose custom ops** into standard ops for portability
5. Build a **library abstraction** for managing families of related custom operators